# Grokking SAE — Sparse Autoencoder on Grokking Transformer
### Colab Notebook

**Basiert auf:**
- Elhage et al. (2022) *Toy Models of Superposition* — transformer-circuits.pub
- Bricken et al. (2023) *Towards Monosemanticity* — transformer-circuits.pub
- Nanda et al. (2023) *Progress Measures for Grokking* — arXiv:2301.05217

**Zentrale Vorhersage:** SAE-Features auf dem post-grokking Modell entsprechen  
den dominanten Fourier-Frequenzen aus der mechanistischen Analyse.

---
**Setup:** Lade `grokking_correct.py` und den `checkpoints/` Ordner  
auf Google Drive hoch, dann passe `DRIVE_PATH` in Zelle 2 an.


In [ ]:
# Zelle 1 — GPU-Check & Imports
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA verfügbar: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Kein GPU — CPU wird verwendet (langsamer)")


In [ ]:
# Zelle 2 — Google Drive mounten & Pfad konfigurieren

from google.colab import drive
drive.mount('/content/drive')

import sys, os

# ── HIER ANPASSEN ──────────────────────────────────────────────
DRIVE_PATH = '/content/drive/MyDrive/Grokking'  # Pfad zu deinem Ordner
# ───────────────────────────────────────────────────────────────

# Pfad registrieren damit grokking_correct.py importierbar ist
if DRIVE_PATH not in sys.path:
    sys.path.insert(0, DRIVE_PATH)
os.chdir(DRIVE_PATH)

# Checkpoints prüfen
from pathlib import Path
ckpt_dir = Path(DRIVE_PATH) / 'checkpoints'
if ckpt_dir.exists():
    ckpts = sorted(ckpt_dir.glob('*.pt'))
    print(f"Checkpoints gefunden: {[c.name for c in ckpts]}")
else:
    print(f"WARNUNG: checkpoints/ nicht gefunden unter {ckpt_dir}")

print(f"Arbeitsverzeichnis: {os.getcwd()}")
print(f"sys.path[0]: {sys.path[0]}")


In [ ]:
# Zelle 3 — Imports & Modell-Definitionen
# Voraussetzung: Zelle 2 muss zuerst ausgeführt worden sein

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass

from grokking_correct import GrokkingTransformer, Config

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

# Output-Ordner auf Drive
OUTPUT_DIR    = Path(DRIVE_PATH) / 'sae_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR   = Path(DRIVE_PATH) / 'checkpoints'
CHECKPOINT_STEPS = [1000, 3000, 7500, 15000, 20000]
print(f"Outputs: {OUTPUT_DIR}")
print("Imports erfolgreich ✓")


In [ ]:
# Zelle 4 — SAE Konfiguration & Architektur

@dataclass
class SAEConfig:
    d_in:            int   = 128    # = d_model des Transformers
    d_sae:           int   = 1024   # 8x d_in — Überparametrisierung
    lr:              float = 1e-3
    l1_coeff:        float = 1e-2   # Startwert — wird durch Sweep optimiert
    num_steps:       int   = 50_000
    batch_size:      int   = 256
    seed:            int   = 42
    activation_site: str   = 'post_ff'


class SparseAutoencoder(nn.Module):
    """
    SAE nach Bricken et al. (2023):
      encode: h = ReLU(W_enc(x - b_pre) + b_enc)
      decode: x̂ = W_dec h + b_pre
      Loss  = ||x - x̂||² + λ ||h||₁
    """
    def __init__(self, cfg: SAEConfig):
        super().__init__()
        self.cfg   = cfg
        self.W_enc = nn.Parameter(torch.empty(cfg.d_in, cfg.d_sae))
        self.b_enc = nn.Parameter(torch.zeros(cfg.d_sae))
        self.W_dec = nn.Parameter(torch.empty(cfg.d_sae, cfg.d_in))
        self.b_pre = nn.Parameter(torch.zeros(cfg.d_in))
        nn.init.kaiming_uniform_(self.W_enc, nonlinearity='relu')
        nn.init.kaiming_uniform_(self.W_dec)
        self._normalize_decoder()

    @torch.no_grad()
    def _normalize_decoder(self):
        norms = self.W_dec.norm(dim=1, keepdim=True).clamp(min=1e-8)
        self.W_dec.data /= norms

    def encode(self, x):
        return F.relu((x - self.b_pre) @ self.W_enc + self.b_enc)

    def decode(self, h):
        return h @ self.W_dec + self.b_pre

    def forward(self, x):
        h = self.encode(x)
        return self.decode(h), h

    def loss(self, x):
        x_hat, h    = self(x)
        recon        = (x - x_hat).pow(2).mean()
        sparsity     = self.cfg.l1_coeff * h.abs().mean()
        return recon + sparsity, recon, sparsity, h

print("SAE Architektur geladen ✓")


In [ ]:
# Zelle 5 — Hilfsfunktionen

def get_all_inputs(cfg):
    pairs  = [(a, b) for a in range(cfg.p) for b in range(cfg.p)]
    inputs = torch.tensor([[a, b, cfg.p] for a, b in pairs], dtype=torch.long)
    labels = torch.tensor([(a + b) % cfg.p for a, b in pairs], dtype=torch.long)
    return inputs, labels, pairs


@torch.no_grad()
def extract_activations(model, inputs, site='post_ff', batch_size=512):
    model.eval()
    all_acts = []
    for start in range(0, len(inputs), batch_size):
        batch = inputs[start:start+batch_size].to(DEVICE)
        B, T  = batch.shape
        pos   = torch.arange(T, device=DEVICE).unsqueeze(0)
        h     = model.tok_emb(batch) + model.pos_emb(pos)
        h2, _ = model.attn(h, h, h)
        h     = model.norm1(h + h2)
        if site == 'post_attn':
            all_acts.append(h[:, -1, :].cpu()); continue
        h = model.norm2(h + model.ff(h))
        all_acts.append(h[:, -1, :].cpu())
    return torch.cat(all_acts, dim=0)


def load_checkpoint(step, cfg):
    path = CHECKPOINT_DIR / f"step_{step:06d}.pt"
    ckpt = torch.load(path, map_location='cpu', weights_only=True)
    model = GrokkingTransformer(cfg).to(DEVICE)
    model.load_state_dict(ckpt['model_state'])
    model.eval()
    return model, ckpt.get('train_acc'), ckpt.get('val_acc')


def train_sae(activations, sae_cfg, verbose=True):
    torch.manual_seed(sae_cfg.seed)
    sae      = SparseAutoencoder(sae_cfg).to(DEVICE)
    optimizer = torch.optim.Adam(sae.parameters(), lr=sae_cfg.lr)
    act_mean = activations.mean(dim=0, keepdim=True)
    act_std  = activations.std(dim=0,  keepdim=True).clamp(min=1e-8)
    acts_n   = (activations - act_mean) / act_std
    loader   = DataLoader(TensorDataset(acts_n), sae_cfg.batch_size, shuffle=True)
    log_int  = sae_cfg.num_steps // 10

    def infinite(loader):
        while True: yield from loader

    step = 0
    for (batch,) in infinite(loader):
        if step >= sae_cfg.num_steps: break
        batch = batch.to(DEVICE)
        total, recon, sparse, h = sae.loss(batch)
        optimizer.zero_grad(); total.backward(); optimizer.step()
        sae._normalize_decoder()
        step += 1
        if verbose and step % log_int == 0:
            dead = (h.abs().max(dim=0).values < 1e-6).sum().item()
            l0   = (h > 0).float().sum(dim=1).mean().item()
            print(f"  step {step:6d} | loss {total.item():.4f} "
                  f"| recon {recon.item():.4f} "
                  f"| dead {dead}/{sae_cfg.d_sae} "
                  f"| L0 {l0:.1f}")

    sae.act_mean = act_mean
    sae.act_std  = act_std
    return sae.cpu()


@torch.no_grad()
def compute_feature_activations(sae, activations):
    acts_n = (activations - sae.act_mean) / sae.act_std
    return sae.encode(acts_n).numpy()


print("Hilfsfunktionen geladen ✓")


In [ ]:
# Zelle 6 — L1-Sweep: optimales λ finden
# Ziel: L0 < 50 aktive Features bei < 30% toten Features

grok_cfg = Config()
sae_cfg  = SAEConfig()

available = [s for s in CHECKPOINT_STEPS
             if (CHECKPOINT_DIR / f"step_{s:06d}.pt").exists()]
print(f"Verfügbare Checkpoints: {available}")

inputs, labels, pairs = get_all_inputs(grok_cfg)
print(f"Input-Raum: {len(inputs)} Paare")

# Aktivierungen auf post-grokking Checkpoint
post_step   = max(available)
model_post, _, _ = load_checkpoint(post_step, grok_cfg)
acts_post   = extract_activations(model_post, inputs, sae_cfg.activation_site)

# Sweep
l1_values   = [1e-3, 5e-3, 1e-2, 5e-2, 1e-1, 2e-1]
sweep_results = {}
sweep_cfg   = SAEConfig(num_steps=10_000)

print(f"\n{'l1':>8}  {'recon':>8}  {'dead%':>8}  {'L0':>8}")
print("─" * 40)

for l1 in l1_values:
    sweep_cfg.l1_coeff = l1
    sae_tmp = train_sae(acts_post, sweep_cfg, verbose=False)
    acts_n  = (acts_post - sae_tmp.act_mean) / sae_tmp.act_std
    with torch.no_grad():
        x_hat, h = sae_tmp(acts_n)
    recon = (acts_n - x_hat).pow(2).mean().item()
    dead  = (h.abs().max(dim=0).values < 1e-6).float().mean().item()
    l0    = (h > 0).float().sum(dim=1).mean().item()
    sweep_results[l1] = dict(recon=recon, dead=dead, l0=l0)
    print(f"{l1:8.1e}  {recon:8.4f}  {dead*100:7.1f}%  {l0:8.1f}")

# Bestes λ wählen
candidates = [l1 for l1, r in sweep_results.items()
              if r['dead'] < 0.30 and r['l0'] < 50]
if not candidates:
    candidates = [l1 for l1, r in sweep_results.items() if r['dead'] < 0.50]
best_l1 = min(candidates, key=lambda l1: sweep_results[l1]['recon'],
              default=sae_cfg.l1_coeff)
sae_cfg.l1_coeff = best_l1
print(f"\nGewähltes λ: {best_l1}")


In [ ]:
# Zelle 7 — SAE Training mit per-Checkpoint λ-Optimierung
# Jeder Checkpoint bekommt sein eigenes optimales λ

def find_best_l1(activations, l1_values=None, num_steps=8_000):
    """Findet bestes λ für gegebene Aktivierungen."""
    if l1_values is None:
        l1_values = [5e-3, 1e-2, 2e-2, 5e-2, 1e-1]
    best_l1  = l1_values[0]
    best_l0  = float('inf')
    target_l0 = 30  # Ziel: ~30 aktive Features von 1024

    tmp_cfg = SAEConfig(num_steps=num_steps)
    results = {}
    for l1 in l1_values:
        tmp_cfg.l1_coeff = l1
        sae_tmp = train_sae(activations, tmp_cfg, verbose=False)
        acts_n  = (activations - sae_tmp.act_mean) / sae_tmp.act_std
        with torch.no_grad():
            _, h = sae_tmp(acts_n)
        dead = (h.abs().max(dim=0).values < 1e-6).float().mean().item()
        l0   = (h > 0).float().sum(dim=1).mean().item()
        results[l1] = dict(dead=dead, l0=l0)

    # Wähle λ das L0 am nächsten an target_l0 bringt bei < 40% dead features
    candidates = [(l1, abs(r['l0'] - target_l0))
                  for l1, r in results.items() if r['dead'] < 0.40]
    if candidates:
        best_l1 = min(candidates, key=lambda x: x[1])[0]

    print(f"  L1-Sweep: ", end='')
    for l1, r in results.items():
        marker = ' ←' if l1 == best_l1 else ''
        print(f"λ={l1:.0e} L0={r['l0']:.0f} dead={r['dead']*100:.0f}%{marker}", end='  ')
    print()
    return best_l1


comparison_results = {}
steps_to_analyze   = [s for s in [1000, 7500, 15000, 20000] if s in available]

for step in steps_to_analyze:
    print(f"\n{'='*55}")
    print(f"Checkpoint Step {step}")
    print(f"{'='*55}")

    model, t_acc, v_acc = load_checkpoint(step, grok_cfg)
    acts = extract_activations(model, inputs, sae_cfg.activation_site)
    print(f"Aktivierungen: {acts.shape}  mean={acts.mean():.3f}  std={acts.std():.3f}")

    # Per-Checkpoint λ bestimmen
    best_l1 = find_best_l1(acts)
    sae_cfg.l1_coeff = best_l1
    print(f"  → Gewähltes λ: {best_l1}")

    # Volles Training
    sae = train_sae(acts, sae_cfg, verbose=True)

    # Fourier-Referenz
    emb       = model.tok_emb.weight[:grok_cfg.p].detach().cpu().numpy()
    fft_emb   = np.abs(np.fft.fft(emb, axis=0)) ** 2
    power_emb = fft_emb.mean(axis=1)
    half      = grok_cfg.p // 2
    top_emb   = list(np.argsort(power_emb[1:half])[-3:] + 1)

    # SAE Feature Fourier
    feat_acts    = compute_feature_activations(sae, acts)
    feat_acts_2d = feat_acts.reshape(grok_cfg.p, grok_cfg.p, -1)
    feat_acts_a  = feat_acts_2d.mean(axis=1)
    fft_feat     = np.abs(np.fft.fft(feat_acts_a, axis=0)) ** 2
    mean_acts_f  = feat_acts.mean(axis=0)
    top_k_idx    = np.argsort(mean_acts_f)[-10:]
    power_feat   = fft_feat[:, top_k_idx].mean(axis=1)
    top_feat     = list(np.argsort(power_feat[1:half])[-3:] + 1)

    p1      = power_emb[1:half]
    p2      = power_feat[1:half]
    cos_sim = float(np.dot(p1, p2) / (np.linalg.norm(p1)*np.linalg.norm(p2)+1e-12))

    # L0 messen
    feat_l0 = (feat_acts > 0).sum(axis=1).mean()

    comparison_results[step] = dict(
        val_acc=v_acc or 0.0, model=model,
        top_emb_freqs=top_emb, top_feat_freqs=top_feat,
        power_emb=power_emb, power_feat=power_feat,
        cos_sim=cos_sim, l1=best_l1, l0=feat_l0,
        sae=sae, acts=acts,
    )
    print(f"  Emb-Freq: {top_emb}  SAE-Freq: {top_feat}  "
          f"CosSim: {cos_sim*100:.0f}%  L0: {feat_l0:.1f}")

print("\nTraining abgeschlossen ✓")


In [ ]:
# Zelle 8 — Einzelplots pro Checkpoint

def plot_sae_fourier_match(sae, activations, model, cfg, step, val_acc, save_path):
    emb           = model.tok_emb.weight[:cfg.p].detach().cpu().numpy()
    fft_emb       = np.abs(np.fft.fft(emb, axis=0)) ** 2
    mean_power_emb = fft_emb.mean(axis=1)
    half          = cfg.p // 2

    feat_acts    = compute_feature_activations(sae, activations)
    mean_acts    = feat_acts.mean(axis=0)
    top_idx      = np.argsort(mean_acts)[-10:]
    feat_2d      = feat_acts.reshape(cfg.p, cfg.p, -1)
    feat_a       = feat_2d.mean(axis=1)
    fft_f        = np.abs(np.fft.fft(feat_a, axis=0)) ** 2
    top_feat_pwr = fft_f[:, top_idx].mean(axis=1)
    l0           = (feat_acts > 0).sum(axis=1)

    acts_n = (activations - sae.act_mean) / sae.act_std
    with torch.no_grad():
        x_hat, _ = sae(acts_n)
    recon_err = (acts_n - x_hat).pow(2).mean(dim=1).numpy().reshape(cfg.p, cfg.p)

    top3_emb  = np.argsort(mean_power_emb[1:half])[-3:] + 1
    top3_feat = np.argsort(top_feat_pwr[1:half])[-3:] + 1
    p1        = mean_power_emb[1:half]
    p2        = top_feat_pwr[1:half]
    cos_sim   = float(np.dot(p1, p2) / (np.linalg.norm(p1)*np.linalg.norm(p2)+1e-12))

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'SAE Analyse — Step {step:,} | val={val_acc*100:.0f}% | '
                 f'd_sae={sae.cfg.d_sae}, λ={sae.cfg.l1_coeff:.0e}',
                 fontsize=13, fontweight='bold')

    ax = axes[0, 0]
    ax.bar(np.arange(1, half), mean_power_emb[1:half], color='#7c6af7', alpha=0.8)
    for k in top3_emb:
        ax.axvline(k, color='#f76a8c', linewidth=2, linestyle='--', alpha=0.8)
        ax.text(k, mean_power_emb[k]*1.05, f'k={k}', ha='center', fontsize=8, color='#f76a8c')
    ax.set_title('Token-Embedding Fourier (Referenz)', fontweight='bold')
    ax.set_xlim(0, half); ax.set_xlabel('Frequenz k')

    ax = axes[0, 1]
    ax.bar(np.arange(1, half), top_feat_pwr[1:half], color='#6af7c2', alpha=0.8)
    for k in top3_feat:
        ax.axvline(k, color='#f76a8c', linewidth=2, linestyle='--', alpha=0.8)
        ax.text(k, top_feat_pwr[k]*1.05, f'k={k}', ha='center', fontsize=8, color='#f76a8c')
    for k in top3_emb:
        ax.axvline(k, color='#7c6af7', linewidth=1, linestyle=':', alpha=0.5)
    ax.set_title(f'Top-10 SAE-Feature Fourier\nSpektrum-Match: {cos_sim*100:.0f}%', fontweight='bold')
    ax.set_xlim(0, half); ax.set_xlabel('Frequenz k')

    ax = axes[1, 0]
    ax.hist(l0, bins=30, color='#f7a46a', alpha=0.8, edgecolor='white')
    ax.axvline(l0.mean(), color='red', linewidth=2, label=f'Mean L0={l0.mean():.1f}')
    ax.set_title('Feature-Sparsität (L0)', fontweight='bold')
    ax.set_xlabel(f'Aktive Features (von {sae.cfg.d_sae})')
    ax.legend()

    ax = axes[1, 1]
    im = ax.imshow(recon_err, cmap='RdYlGn_r', aspect='auto')
    ax.set_title('Rekonstruktionsfehler pro (a,b)', fontweight='bold')
    ax.set_xlabel('b'); ax.set_ylabel('a')
    plt.colorbar(im, ax=ax, label='MSE')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Gespeichert: {save_path}")

for step, r in comparison_results.items():
    label = 'pre' if r['val_acc'] < 0.5 else 'post'
    plot_sae_fourier_match(
        r['sae'], r['acts'], r['model'], grok_cfg, step, r['val_acc'],
        save_path=str(OUTPUT_DIR / f'sae_features_{label}_step{step}.png')
    )


In [ ]:
# Zelle 9 — Feature-Aktivierungsgitter (post-grokking)

best_step = max(comparison_results.keys())
r         = comparison_results[best_step]
feat_acts = compute_feature_activations(r['sae'], r['acts'])
feat_2d   = feat_acts.reshape(grok_cfg.p, grok_cfg.p, -1)
mean_acts = feat_acts.mean(axis=0)
top_idx   = np.argsort(mean_acts)[-6:][::-1]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()
fig.suptitle(f'SAE Feature-Aktivierungen über (a,b) Raum — Step {best_step:,}',
             fontsize=13, fontweight='bold')

for plot_i, feat_i in enumerate(top_idx):
    ax   = axes[plot_i]
    grid = feat_2d[:, :, feat_i]
    vmax = np.abs(grid).max()
    im   = ax.imshow(grid, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax)
    ax.set_title(f'Feature {feat_i} | mean={mean_acts[feat_i]:.3f}', fontsize=9)
    ax.set_xlabel('b'); ax.set_ylabel('a')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
save_path = str(OUTPUT_DIR / 'sae_activation_grid.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Gespeichert: {save_path}")


In [ ]:
# Zelle 10 — Übersichtsplot: pre vs post Grokking

steps = sorted(comparison_results.keys())
half  = grok_cfg.p // 2
fig, axes = plt.subplots(2, len(steps), figsize=(5*len(steps), 8))
fig.suptitle('SAE Fourier-Features: pre vs post Grokking', fontsize=13, fontweight='bold')

for j, step in enumerate(steps):
    r          = comparison_results[step]
    power_emb  = r['power_emb']
    power_feat = r['power_feat']
    top_emb    = r['top_emb_freqs']
    top_feat   = r['top_feat_freqs']
    cos_sim    = r['cos_sim']

    ax = axes[0, j]
    ax.bar(np.arange(1, half), power_emb[1:half], color='#7c6af7', alpha=0.8)
    for k in top_emb:
        ax.axvline(k, color='#f76a8c', linewidth=2, linestyle='--', alpha=0.8)
    ax.set_title(f'Step {step:,} | val={r["val_acc"]*100:.0f}%\nEmbedding Fourier',
                 fontsize=9, fontweight='bold')
    ax.set_xlim(0, half)
    if j == 0: ax.set_ylabel('Embedding Fourier Power', fontsize=9)

    ax = axes[1, j]
    ax.bar(np.arange(1, half), power_feat[1:half], color='#6af7c2', alpha=0.8)
    for k in top_feat:
        ax.axvline(k, color='#f76a8c', linewidth=2, linestyle='--', alpha=0.8)
    for k in top_emb:
        ax.axvline(k, color='#7c6af7', linewidth=1, linestyle=':', alpha=0.5)
    col = '#4caf7d' if cos_sim >= 0.60 else '#e05a6a'
    ax.set_title(f'SAE Feature Fourier\nSpektrum-Match: {cos_sim*100:.0f}%',
                 fontsize=9, fontweight='bold', color=col)
    ax.set_xlim(0, half)
    if j == 0: ax.set_ylabel('SAE Feature Fourier Power', fontsize=9)

plt.tight_layout()
save_path = str(OUTPUT_DIR / 'sae_comparison.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Gespeichert: {save_path}")


In [ ]:
# Zelle 11 — Zusammenfassung

print("=" * 65)
print("ERGEBNISSE — SAE Fourier-Feature Analyse")
print("=" * 65)
print(f"\n{'Step':>8}  {'Val Acc':>8}  {'Emb Freqs':>12}  {'SAE Freqs':>12}  {'CosSim':>8}")
print("─" * 60)
for step, r in sorted(comparison_results.items()):
    print(f"{step:8d}  {r['val_acc']*100:7.1f}%  "
          f"{str(r['top_emb_freqs']):>12}  "
          f"{str(r['top_feat_freqs']):>12}  "
          f"{r['cos_sim']*100:6.1f}%")

print(f"\nOutputs gespeichert in: {OUTPUT_DIR}")
print("  sae_features_pre_step*.png")
print("  sae_features_post_step*.png")
print("  sae_activation_grid.png")
print("  sae_comparison.png")


In [ ]:
# Zelle 12 — Superposition-Analyse
# ====================================================
# Frage: Lernt der SAE dedizierte Features pro Fourier-Frequenz
#        (kein Superposition) oder teilen sich Features mehrere
#        Frequenzen (Superposition)?
#
# Methode:
#   1. Für jede dominante Embedding-Frequenz k: berechne wie stark
#      jedes SAE-Feature diese Frequenz kodiert (via cos/sin Projektion)
#   2. Visualisiere top-3 Features pro Frequenz
#   3. Messe Exklusivität: hat jedes Feature eine dominante Frequenz
#      oder trägt es zu mehreren gleichzeitig bei?

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

def analyze_superposition(sae, activations, cfg, step, val_acc,
                          target_freqs=None, save_path=None):
    """
    Hauptanalyse: Superposition vs. dedizierte Features.

    target_freqs: Liste von Fourier-Frequenzen die untersucht werden
                  (aus Embedding-Analyse bekannt)
    """
    if target_freqs is None:
        # Aus Embedding-Analyse für diesen Checkpoint bestimmen
        model_ref = comparison_results[step]['model']
        emb       = model_ref.tok_emb.weight[:cfg.p].detach().cpu().numpy()
        fft_emb   = np.abs(np.fft.fft(emb, axis=0)) ** 2
        power_emb = fft_emb.mean(axis=1)
        half      = cfg.p // 2
        target_freqs = list(np.argsort(power_emb[1:half])[-3:] + 1)

    x        = np.arange(cfg.p)
    d_sae    = sae.cfg.d_sae

    # ── Schritt 1: Fourier-Responsivität jedes SAE-Features ──────
    # W_dec[i] ist die Feature-Richtung im d_model Raum
    # Wir messen wie stark Feature i auf Frequenz k reagiert indem
    # wir die Feature-Aktivierungen über (a,?) als Funktion von a betrachten
    # und FFT anwenden

    feat_acts    = compute_feature_activations(sae, activations)  # [p*p, d_sae]
    feat_acts_2d = feat_acts.reshape(cfg.p, cfg.p, -1)            # [p, p, d_sae]
    feat_acts_a  = feat_acts_2d.mean(axis=1)                      # [p, d_sae]
    feat_acts_b  = feat_acts_2d.mean(axis=0)                      # [p, d_sae]

    # FFT über a-Dimension für jedes Feature
    fft_a = np.abs(np.fft.fft(feat_acts_a, axis=0)) ** 2         # [p, d_sae]
    fft_b = np.abs(np.fft.fft(feat_acts_b, axis=0)) ** 2         # [p, d_sae]
    fft_combined = (fft_a + fft_b) / 2                            # [p, d_sae]

    # ── Schritt 2: Pro Frequenz k — top-3 Features ───────────────
    # feature_freq_power[k, i] = wie stark Feature i auf Frequenz k reagiert
    freq_power = fft_combined                                      # [p, d_sae]

    # ── Schritt 3: Exklusivitäts-Analyse ─────────────────────────
    # Für jedes Feature: was ist die dominante Frequenz?
    # Exklusivitäts-Score = Power(dominant_freq) / Power(alle_freqs)
    half     = cfg.p // 2
    fp_half  = freq_power[1:half, :]                              # [half-1, d_sae]
    dom_freq = fp_half.argmax(axis=0) + 1                         # [d_sae] — dominante Freq
    dom_pow  = fp_half.max(axis=0)                                 # [d_sae]
    tot_pow  = fp_half.sum(axis=0).clip(min=1e-8)                 # [d_sae]
    exclusivity = dom_pow / tot_pow                                # [d_sae], 1.0 = exklusiv

    # Nur aktive Features berücksichtigen (mean activation > threshold)
    mean_acts = feat_acts.mean(axis=0)                             # [d_sae]
    active    = mean_acts > mean_acts.mean()                       # [d_sae] bool

    # ── Plot ─────────────────────────────────────────────────────
    n_freqs  = len(target_freqs)
    fig      = plt.figure(figsize=(5 * n_freqs + 4, 14))
    gs       = fig.add_gridspec(3, n_freqs + 1, hspace=0.4, wspace=0.35)
    fig.suptitle(f'Superposition-Analyse — Step {step:,} | val={val_acc*100:.0f}%\n'
                 f'Frage: Lernt der SAE dedizierte Features pro Fourier-Frequenz?',
                 fontsize=13, fontweight='bold')

    colors = ['#7c6af7', '#f76a8c', '#6af7c2']

    for col, (k, col_color) in enumerate(zip(target_freqs, colors)):
        # Top-3 Features für diese Frequenz (unter aktiven)
        freq_resp = freq_power[k, :]                               # [d_sae]
        freq_resp_active = freq_resp * active.astype(float)
        top3_idx  = np.argsort(freq_resp_active)[-3:][::-1]

        for row, feat_i in enumerate(top3_idx):
            ax = fig.add_subplot(gs[row, col])

            # Aktivierungsgitter für dieses Feature
            grid = feat_acts_2d[:, :, feat_i]
            vmax = np.abs(grid).max()
            im   = ax.imshow(grid, cmap='RdBu_r', aspect='auto',
                             vmin=-vmax, vmax=vmax)

            # Exklusivität berechnen
            excl = exclusivity[feat_i]
            dom_k = dom_freq[feat_i]

            # Titel: Frequenz-Exklusivität anzeigen
            title_color = '#4caf7d' if excl > 0.5 else '#e05a6a'
            ax.set_title(
                f'k={k} → Feature {feat_i}\n'
                f'Dom.Freq=k{dom_k}, Exkl.={excl:.2f}',
                fontsize=8, color=title_color, fontweight='bold'
            )
            ax.set_xlabel('b', fontsize=7); ax.set_ylabel('a', fontsize=7)
            ax.tick_params(labelsize=6)

            if row == 0:
                ax.set_title(
                    f'Frequenz k={k}\nFeature {feat_i} | Exkl.={excl:.2f}',
                    fontsize=9, color=col_color, fontweight='bold'
                )

    # ── Rechte Spalte: Exklusivitäts-Histogramm ──────────────────
    ax_hist = fig.add_subplot(gs[:, -1])
    excl_active = exclusivity[active]
    ax_hist.hist(excl_active, bins=30, color='#7c6af7', alpha=0.8,
                 edgecolor='white', orientation='horizontal')
    ax_hist.axhline(0.5, color='#f76a8c', linewidth=2, linestyle='--',
                    label='0.5 Schwelle')
    ax_hist.axhline(excl_active.mean(), color='gray', linewidth=1.5,
                    linestyle=':', label=f'Mean={excl_active.mean():.2f}')
    frac_dedicated = (excl_active > 0.5).mean()
    ax_hist.set_ylabel('Exklusivität (1.0 = eine Frequenz)', fontsize=9)
    ax_hist.set_xlabel('Anzahl Features', fontsize=9)
    ax_hist.set_title(
        f'Feature-Exklusivität\n'
        f'{frac_dedicated*100:.0f}% der aktiven Features\n'
        f'sind auf eine Frequenz spezialisiert',
        fontsize=9, fontweight='bold'
    )
    ax_hist.legend(fontsize=8)
    ax_hist.set_ylim(0, 1)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Gespeichert: {save_path}")

    # ── Zusammenfassung ──────────────────────────────────────────
    print(f"\nSuperposition-Analyse Step {step:,}:")
    print(f"  Aktive Features: {active.sum()} von {d_sae}")
    print(f"  Mean Exklusivität: {excl_active.mean():.3f}")
    print(f"  Features mit Exkl. > 0.5: {(excl_active > 0.5).sum()} "
          f"({frac_dedicated*100:.0f}%)")
    print(f"  Features mit Exkl. > 0.8: {(excl_active > 0.8).sum()} "
          f"({(excl_active > 0.8).mean()*100:.0f}%)")
    print()
    print(f"  Top-3 Features pro Zielfrequenz:")
    for k, col_color in zip(target_freqs, colors):
        freq_resp_active = freq_power[k, :] * active.astype(float)
        top3 = np.argsort(freq_resp_active)[-3:][::-1]
        for feat_i in top3:
            excl = exclusivity[feat_i]
            dom_k = dom_freq[feat_i]
            marker = '✓ dediziert' if excl > 0.5 else '⚠ Superposition'
            print(f"    k={k} → Feature {feat_i}: "
                  f"Exkl.={excl:.2f}, dom.Freq=k{dom_k} [{marker}]")

    return exclusivity, dom_freq, active


# ── Analyse auf Step 15000 (bester Match) ────────────────────────
print("="*60)
print("Superposition-Analyse auf Step 15,000")
print("="*60)

step_key = 15000
r        = comparison_results[step_key]

exclusivity_15k, dom_freq_15k, active_15k = analyze_superposition(
    r['sae'], r['acts'], grok_cfg,
    step=step_key, val_acc=r['val_acc'],
    target_freqs=r['top_emb_freqs'],
    save_path=str(OUTPUT_DIR / 'sae_superposition_step15000.png')
)

# ── Vergleich pre vs post ─────────────────────────────────────────
print("\n" + "="*60)
print("Superposition-Analyse auf Step 1,000 (pre-grokking)")
print("="*60)

step_key_pre = 1000
r_pre        = comparison_results[step_key_pre]

exclusivity_1k, dom_freq_1k, active_1k = analyze_superposition(
    r_pre['sae'], r_pre['acts'], grok_cfg,
    step=step_key_pre, val_acc=r_pre['val_acc'],
    target_freqs=r_pre['top_emb_freqs'],
    save_path=str(OUTPUT_DIR / 'sae_superposition_step1000.png')
)

# ── Direkter Vergleich ────────────────────────────────────────────
print("\n" + "="*60)
print("Direkter Vergleich: Pre vs Post Grokking")
print("="*60)

excl_pre  = exclusivity_1k[active_1k]
excl_post = exclusivity_15k[active_15k]

print(f"  Pre-grokking  (Step 1k):  Mean Exkl. = {excl_pre.mean():.3f}, "
      f"Dediziert (>0.5): {(excl_pre > 0.5).mean()*100:.0f}%")
print(f"  Post-grokking (Step 15k): Mean Exkl. = {excl_post.mean():.3f}, "
      f"Dediziert (>0.5): {(excl_post > 0.5).mean()*100:.0f}%")

# Zusammenfassender Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Superposition: Pre vs Post Grokking\n'
             'Exklusivität = Anteil der Feature-Power auf dominanter Frequenz',
             fontsize=12, fontweight='bold')

for ax, excl, label, color in [
    (axes[0], excl_pre,  f'Pre-Grokking (Step 1k)\nval=30%',  '#e05a6a'),
    (axes[1], excl_post, f'Post-Grokking (Step 15k)\nval=100%', '#4caf7d'),
]:
    ax.hist(excl, bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(0.5, color='black', linewidth=2, linestyle='--',
               label='Superposition | Dediziert')
    ax.axvline(excl.mean(), color='gray', linewidth=1.5, linestyle=':',
               label=f'Mean={excl.mean():.2f}')
    frac = (excl > 0.5).mean()
    ax.set_title(f'{label}\n{frac*100:.0f}% dedizierte Features',
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Exklusivität (0=Superposition, 1=dediziert)', fontsize=10)
    ax.set_ylabel('Anzahl Features', fontsize=10)
    ax.legend(fontsize=9)
    ax.set_xlim(0, 1)

plt.tight_layout()
save_path = str(OUTPUT_DIR / 'sae_superposition_comparison.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"\nGespeichert: {save_path}")
